# Notebook 06: VIX/VXX Volatility Modeling with Continuous HMM

This notebook applies the continuous Hidden Markov Model (CHMM) framework to the **CBOE Volatility Index (VIX)**,
the market's benchmark measure of expected 30-day implied volatility derived from S&P 500 index options.

### Why model VIX with a regime-switching model?

The VIX exhibits several properties that make it a natural candidate for CHMM modeling:

1. **Mean reversion** -- VIX tends to revert to a long-run level (~18--20), but the speed and target differ across market regimes (calm vs crisis).
2. **Volatility clustering** -- Periods of elevated VIX persist (e.g., 2008 GFC, 2020 COVID), consistent with latent regime dynamics.
3. **Heavy-tailed log returns** -- VIX daily changes are leptokurtic, with extreme spikes that a single Gaussian cannot capture.
4. **Asymmetric behavior** -- VIX rises faster during sell-offs than it falls during recoveries (the "leverage effect" in vol space).

A CHMM with $K$ hidden states can decompose the VIX return distribution into regime-specific Gaussians,
capturing these features through a mixture of calm, moderate, and crisis regimes.

### VXX: A Related Instrument

The **iPath Series B S&P 500 VIX Short-Term Futures ETN (VXX)** tracks the daily return of a rolling portfolio
of first- and second-month VIX futures contracts. Unlike VIX itself (which is not directly tradeable), VXX provides
a tradeable exposure to volatility. VXX is subject to contango drag (futures curve roll cost) and exhibits a
structural negative drift. The same CHMM framework applies to VXX returns -- we discuss this in the final section.

### Sections
1. Setup and configuration
2. VIX data loading and return computation
3. VIX stylized facts (distribution, Q-Q, ACF)
4. CHMM training on VIX returns
5. Emission distributions and regime decoding
6. Monte Carlo simulation and in-sample validation
7. GARCH(1,1) comparison
8. VIX regime-to-equity volatility map
9. Out-of-sample validation
10. VXX discussion

## Setup

In [ ]:
include("../Include.jl");

## Configuration

Tune the parameters below. VIX returns are computed as annualized excess log returns
using the close price (VIX data does not include `volume_weighted_average_price`).

In [ ]:
# --- USER-TUNABLE PARAMETERS ---
K = 6;                  # number of hidden states
MAX_ITER = 60;          # maximum Baum-Welch iterations
N_PATHS = 500;          # Monte Carlo paths for validation
L = 252;                # ACF max lag (1 trading year)
Δt = 1/252;             # daily time step in years
ticker = "VIX";         # volatility index

## Load VIX Data

The VIX training set covers ~20 years up to December 31, 2024.
The out-of-sample set starts January 2, 2025.
VIX DataFrames have columns: `date`, `open`, `high`, `low`, `close`, `volume`.
We use `keycol=:close` since there is no `volume_weighted_average_price`.

In [ ]:
# --- LOAD VIX DATA ---
vix_train = MyVolatilityDataSet()["dataset"];
R_vix_is = log_growth_matrix(vix_train, "VIX"; Δt=Δt, keycol=:close);

vix_oos = MyOutOfSampleVolatilityDataSet()["dataset"];
R_vix_oos = log_growth_matrix(vix_oos, "VIX"; Δt=Δt, keycol=:close);

println("VIX In-Sample: T = $(length(R_vix_is)) observations");
println("VIX Out-of-Sample: T = $(length(R_vix_oos)) observations");
println("IS mean = $(round(mean(R_vix_is), digits=2)), std = $(round(std(R_vix_is), digits=2))");
println("OoS mean = $(round(mean(R_vix_oos), digits=2)), std = $(round(std(R_vix_oos), digits=2))");

## VIX Stylized Facts

We characterize VIX daily log returns with the same four-panel diagnostic used for equities:
- **(a)** Marginal distribution with Gaussian and Laplace fits
- **(b)** Normal Q-Q plot (heavy tails visible as departures from the 45-degree line)
- **(c)** ACF of raw returns $G_t$
- **(d)** ACF of $|G_t|$ (volatility clustering in vol-of-vol)

In [ ]:
# --- DESCRIPTIVE STATISTICS ---
n_is = length(R_vix_is);
μ_is = mean(R_vix_is);
σ_is = std(R_vix_is);
z_is = (R_vix_is .- μ_is) ./ σ_is;
skew_is = sum(z_is .^ 3) / n_is;
kurt_is = sum(z_is .^ 4) / n_is - 3.0;

# Jarque-Bera
jb_stat = (n_is / 6) * (skew_is^2 + (kurt_is^2) / 4);

# Ljung-Box
lb_lags = 20;
acf_raw_lb = autocor(R_vix_is, 1:lb_lags);
lb_raw = n_is * (n_is + 2) * sum(acf_raw_lb .^ 2 ./ (n_is .- (1:lb_lags)));
acf_abs_lb = autocor(abs.(R_vix_is), 1:lb_lags);
lb_abs = n_is * (n_is + 2) * sum(acf_abs_lb .^ 2 ./ (n_is .- (1:lb_lags)));

println("\n--- VIX In-Sample Descriptive Statistics (T = $n_is) ---");
println("Mean (annualized):        $(round(μ_is, digits=2))");
println("Std Dev (annualized):     $(round(σ_is, digits=2))");
println("Skewness:                 $(round(skew_is, digits=3))");
println("Excess Kurtosis:          $(round(kurt_is, digits=3))");
println("JB statistic:             $(round(jb_stat, digits=1)) (critical ~5.99 at α=0.05)");
println("LB on Gₜ  (lag $lb_lags):       $(round(lb_raw, digits=1)) (critical ~31.4 at α=0.05)");
println("LB on |Gₜ| (lag $lb_lags):      $(round(lb_abs, digits=1)) (critical ~31.4 at α=0.05)");

In [ ]:
# --- FIGURE: VIX STYLIZED FACTS (4-panel, 2x2) ---

# Fit Gaussian
d_gauss = Normal(mean(R_vix_is), std(R_vix_is));

# Fit Laplace via MLE
μ_lap = median(R_vix_is);
b_lap = mean(abs.(R_vix_is .- μ_lap));
d_laplace = Laplace(μ_lap, b_lap);

# Percentile-based x-limits
xl_lo = quantile(R_vix_is, 0.001);
xl_hi = quantile(R_vix_is, 0.999);
x_grid = range(xl_lo, xl_hi, length=1000);

# Panel (a): Marginal distribution
p1 = histogram(R_vix_is, normalize=true, bins=150, alpha=0.4, color=:gray, label="Observed",
    title="(a) Marginal Distribution", titlefontsize=10,
    xlabel="Excess Growth Rate", ylabel="Density");
plot!(p1, x_grid, pdf.(d_gauss, x_grid), lw=2, color=:blue, ls=:dash, label="Gaussian");
plot!(p1, x_grid, pdf.(d_laplace, x_grid), lw=2, color=:red, label="Laplace");
xlims!(p1, xl_lo, xl_hi);

# Panel (b): Normal Q-Q plot
sorted_R = sort(R_vix_is);
n_obs = length(sorted_R);
theoretical_q = [quantile(d_gauss, (i - 0.5) / n_obs) for i in 1:n_obs];

p2 = scatter(theoretical_q, sorted_R, ms=1, alpha=0.5, color=:steelblue, label="",
    title="(b) Normal Q-Q Plot", titlefontsize=10,
    xlabel="Theoretical Quantiles", ylabel="Sample Quantiles");
qq_lo = min(minimum(theoretical_q), minimum(sorted_R));
qq_hi = max(maximum(theoretical_q), maximum(sorted_R));
plot!(p2, [qq_lo, qq_hi], [qq_lo, qq_hi], lw=2, color=:red, ls=:dash, label="45° line");

# Panel (c): ACF of raw returns
τ = 1:(L-1);
ci_99 = 2.576 / sqrt(n_obs);
acf_raw = autocor(R_vix_is, τ);

p3 = plot(τ, acf_raw, linetype=:sticks, lw=1.5, color=:steelblue, label="ACF(Gₜ)",
    title="(c) Returns Autocorrelation", titlefontsize=10,
    xlabel="Lag (trading days)", ylabel="ACF");
hline!(p3, [ci_99, -ci_99], lw=1.5, color=:gray, ls=:dash, label="99% CI");
hline!(p3, [0.0], lw=0.5, color=:black, label="");

# Panel (d): ACF of |returns|
acf_abs = autocor(abs.(R_vix_is), τ);

p4 = plot(τ, acf_abs, linetype=:sticks, lw=1.5, color=:darkorange, label="ACF(|Gₜ|)",
    title="(d) Volatility Clustering (Vol-of-Vol)", titlefontsize=10,
    xlabel="Lag (trading days)", ylabel="ACF");
hline!(p4, [ci_99, -ci_99], lw=1.5, color=:gray, ls=:dash, label="99% CI");
hline!(p4, [0.0], lw=0.5, color=:black, label="");

# Combine into 2x2
fig1 = plot(p1, p2, p3, p4, layout=(2,2), size=(1000, 700),
    plot_title="VIX Stylized Facts: Daily Excess Log Returns (In-Sample)",
    plot_titlefontsize=12);
display(fig1);

savefig(fig1, joinpath(_PATH_TO_FIGURES, "Fig-VIX-Stylized-Facts-IS.svg"));
savefig(fig1, joinpath(_PATH_TO_FIGURES, "Fig-VIX-Stylized-Facts-IS.pdf"));
println("Saved VIX stylized facts figure to figs/");

## Train CHMM on VIX Returns

We train a $K$-state continuous Gaussian HMM on VIX daily log returns using the Baum-Welch algorithm.
Quantile-based initialization prevents degenerate solutions.

In [ ]:
# --- TRAIN CHMM ---
model = build(MyContinuousHiddenMarkovModel, (
    observations = R_vix_is,
    number_of_states = K,
    max_iter = MAX_ITER
));

println("Training complete: $(length(model.log_likelihood_history)) EM iterations");
println("Final log-likelihood: $(round(model.log_likelihood_history[end], digits=2))");

In [ ]:
# --- CONVERGENCE PLOT ---
p_conv = plot(model.log_likelihood_history,
    title="Baum-Welch Convergence (VIX, K=$K)",
    xlabel="Iteration", ylabel="Log-Likelihood",
    legend=false, lw=2, color=:navy, marker=:circle, ms=3,
    size=(700, 400));
display(p_conv);

savefig(p_conv, joinpath(_PATH_TO_FIGURES, "Fig-VIX-BW-Convergence-K$(K).svg"));

## Emission Distributions

Each hidden state $k$ emits observations from a Gaussian $\mathcal{N}(\mu_k, \sigma_k)$.
The emission PDFs show how the CHMM decomposes VIX returns into regime-specific components.

In [ ]:
# --- EMISSION PDFs ---
p_emit = plot_emission_pdfs(model, ticker);
display(p_emit);

savefig(p_emit, joinpath(_PATH_TO_FIGURES, "Fig-VIX-Emission-PDFs-K$(K).svg"));

In [ ]:
# --- EMISSION PARAMETERS TABLE ---
T_mat = zeros(K, K);
for i in 1:K
    T_mat[i, :] = probs(model.transition[i]);
end
π_stat = (T_mat^1000)[1, :];

println("State | Mean (μ)     | Std Dev (σ)  | π_stationary | Residence (days)");
println("------+--------------+--------------+--------------+-----------------");
for s in 1:K
    d = model.emission[s];
    μ_s = round(mean(d), digits=2);
    σ_s = round(std(d), digits=2);
    π_s = round(π_stat[s], digits=4);
    τ_s = round(1.0 / (1.0 - T_mat[s, s]), digits=2);
    println("  $(lpad(s,2))  | $(lpad(μ_s,12)) | $(lpad(σ_s,12)) | $(lpad(π_s,12)) | $(lpad(τ_s,15))");
end
println("\nSum of π: $(round(sum(π_stat), digits=6))");

## Regime Decoding: Viterbi Overlay on VIX Levels

The Viterbi algorithm decodes the most likely hidden state sequence.
Overlaying regimes on the VIX close price time series reveals how the model
partitions calm, moderate, and crisis periods.

In [ ]:
# --- VITERBI DECODING ---
decoded_states = viterbi(R_vix_is, model);

# VIX close prices (aligned with returns: skip first day)
vix_close = vix_train["VIX"].close[2:end];
vix_dates = vix_train["VIX"].date[2:end];

# Use the built-in regime overlay visualization
p_regime = plot_regime_overlay(vix_dates, Float64.(vix_close), decoded_states, ticker;
    title_text="VIX Regime Overlay (CHMM, K=$K)");
xlabel!(p_regime, "Date");
display(p_regime);

savefig(p_regime, joinpath(_PATH_TO_FIGURES, "Fig-VIX-Regime-Overlay-K$(K).svg"));
savefig(p_regime, joinpath(_PATH_TO_FIGURES, "Fig-VIX-Regime-Overlay-K$(K).pdf"));
println("Saved regime overlay to figs/");

## Monte Carlo Simulation and In-Sample Validation

We simulate `N_PATHS` paths from the trained CHMM (same length as the in-sample data)
and compute distributional validation metrics.

In [ ]:
# --- STATIONARY DISTRIBUTION FOR START STATE ---
start_dist = Categorical(π_stat);
n_steps = length(R_vix_is);

# --- SIMULATE N_PATHS PATHS ---
decoded_is = Array{Float64,2}(undef, n_steps, N_PATHS);
for i in 1:N_PATHS
    s0 = rand(start_dist);
    states = model(s0, n_steps);
    for j in 1:n_steps
        decoded_is[j, i] = rand(model.emission[states[j]]);
    end
end

println("CHMM: $(N_PATHS) paths of length $(n_steps) simulated.");

In [ ]:
# --- VALIDATION METRICS ---
function compute_vix_metrics(observed::Vector{Float64}, simulated_archive::Matrix{Float64};
                             α=0.05, L_acf=252)

    n_paths = size(simulated_archive, 2);
    n_obs = length(observed);

    # Observed targets
    μ_obs = mean(observed); σ_obs = std(observed);
    kurt_obs = sum(((observed .- μ_obs) ./ σ_obs).^4) / n_obs - 3.0;
    τ_range = 1:min(L_acf, n_obs - 1);
    acf_obs = autocor(abs.(observed), τ_range);

    # Accumulators
    ks_pass = 0;
    kurt_sum = 0.0;
    acf_mae_sum = 0.0;

    for i in 1:n_paths
        sim = simulated_archive[:, i];

        # KS test
        ks_pval = pvalue(ApproximateTwoSampleKSTest(observed, sim));
        if ks_pval > α; ks_pass += 1; end

        # Kurtosis
        μ_s = mean(sim); σ_s = std(sim);
        kurt_sum += sum(((sim .- μ_s) ./ σ_s).^4) / length(sim) - 3.0;

        # ACF-MAE
        acf_sim = autocor(abs.(sim), τ_range);
        acf_mae_sum += mean(abs.(acf_obs .- acf_sim));
    end

    return (
        ks_rate     = round(100.0 * ks_pass / n_paths, digits=1),
        kurtosis    = round(kurt_sum / n_paths, digits=2),
        kurt_obs    = round(kurt_obs, digits=2),
        acf_mae     = round(acf_mae_sum / n_paths, digits=4),
    );
end;

metrics_chmm = compute_vix_metrics(R_vix_is, decoded_is; L_acf=L);

println("\n" * "="^55);
println("  CHMM In-Sample Validation: VIX (K=$K)");
println("="^55);
println("  KS pass rate (α=0.05):  $(metrics_chmm.ks_rate)%");
println("  Excess kurtosis (sim):  $(metrics_chmm.kurtosis)");
println("  Excess kurtosis (obs):  $(metrics_chmm.kurt_obs)");
println("  ACF-MAE |Gₜ|:          $(metrics_chmm.acf_mae)");
println("="^55);

## Density and ACF Comparison: Observed vs CHMM Simulated

Visual overlay comparing the marginal density and absolute-return ACF of observed VIX returns
against CHMM-simulated returns.

In [ ]:
# --- FIGURE: DENSITY + ACF COMPARISON ---

# Panel (a): Marginal density
p_dens = plot(title="(a) Marginal Density (IS KS: $(metrics_chmm.ks_rate)%)",
    titlefontsize=9, xlabel="Excess Growth Rate", ylabel="Density");
histogram!(p_dens, R_vix_is, normalize=true, bins=150, alpha=0.3, color=:gray, label="Observed");
density!(p_dens, vec(decoded_is), lw=2, color=:blue, alpha=0.7, label="CHMM (K=$K)");
xlims!(p_dens, quantile(R_vix_is, 0.001), quantile(R_vix_is, 0.999));

# Panel (b): ACF(|G|) comparison
τ_acf = 1:L;
acf_obs_abs = autocor(abs.(R_vix_is), τ_acf);

n_acf_paths = min(200, N_PATHS);
acf_archive = hcat([autocor(abs.(decoded_is[:, i]), τ_acf) for i in 1:n_acf_paths]...);
acf_mean = mean(acf_archive, dims=2)[:];
acf_p10 = [quantile(acf_archive[t, :], 0.10) for t in 1:L];
acf_p90 = [quantile(acf_archive[t, :], 0.90) for t in 1:L];

p_acf = plot(τ_acf, acf_obs_abs, lw=2, color=:red, ls=:dash, label="Observed",
    title="(b) ACF(|Gₜ|) -- Volatility Clustering", titlefontsize=9,
    xlabel="Lag (trading days)", ylabel="ACF(|Gₜ|)");
plot!(p_acf, τ_acf, acf_mean, lw=2, color=:blue, label="CHMM mean");
plot!(p_acf, τ_acf, acf_p10, fillrange=acf_p90, alpha=0.15, color=:blue, label="10-90th pctl");

# Combine
fig_comp = plot(p_dens, p_acf, layout=(1, 2), size=(1200, 400),
    plot_title="VIX: Observed vs CHMM Simulated (K=$K)",
    plot_titlefontsize=12);
display(fig_comp);

savefig(fig_comp, joinpath(_PATH_TO_FIGURES, "Fig-VIX-IS-Comparison-K$(K).svg"));
savefig(fig_comp, joinpath(_PATH_TO_FIGURES, "Fig-VIX-IS-Comparison-K$(K).pdf"));

## GARCH(1,1) Comparison

We fit a GARCH(1,1) model to VIX returns as a single-regime benchmark.
GARCH captures volatility clustering via conditional variance dynamics but lacks
discrete regime structure.

In [ ]:
# --- FIT GARCH(1,1) ---
garch_model = build(MyGARCHModel, (observations = R_vix_is,));

println("GARCH(1,1) Parameters:");
println("  ω = $(round(garch_model.ω, digits=4))");
println("  α = $(round(garch_model.α, digits=4))");
println("  β = $(round(garch_model.β, digits=4))");
println("  μ = $(round(garch_model.μ, digits=4))");
println("  α + β = $(round(garch_model.α + garch_model.β, digits=4)) (persistence)");
println("  Log-likelihood: $(round(garch_model.log_likelihood, digits=2))");

In [ ]:
# --- SIMULATE GARCH PATHS ---
decoded_garch = Array{Float64,2}(undef, n_steps, N_PATHS);
for i in 1:N_PATHS
    decoded_garch[:, i] = simulate_garch(garch_model, n_steps);
end

# Compute metrics
metrics_garch = compute_vix_metrics(R_vix_is, decoded_garch; L_acf=L);

# --- COMPARISON TABLE ---
println("\n" * "="^60);
println("  Model Comparison: CHMM vs GARCH(1,1) on VIX");
println("="^60);
println(rpad("Metric", 25) * rpad("CHMM (K=$K)", 18) * rpad("GARCH(1,1)", 18));
println("-"^60);
println(rpad("KS pass rate (%)", 25) * rpad(metrics_chmm.ks_rate, 18) * rpad(metrics_garch.ks_rate, 18));
println(rpad("Kurtosis (sim)", 25) * rpad(metrics_chmm.kurtosis, 18) * rpad(metrics_garch.kurtosis, 18));
println(rpad("Kurtosis (obs)", 25) * rpad(metrics_chmm.kurt_obs, 18) * rpad(metrics_garch.kurt_obs, 18));
println(rpad("ACF-MAE |Gₜ|", 25) * rpad(metrics_chmm.acf_mae, 18) * rpad(metrics_garch.acf_mae, 18));
println("="^60);
println("Paths: $N_PATHS | Path length: $n_steps | CHMM MAX_ITER: $MAX_ITER");

## VIX Regime Volatility Map: Bridging VIX Regimes to Equity Volatility

One of the key applications of VIX regime modeling is mapping CHMM states to equity volatility.
For each hidden state $s$, we compute the median VIX level among observations decoded to that state:

$$\sigma_s = \frac{\text{median}(\text{VIX}_{\text{close}} \mid \text{state} = s)}{100}$$

This gives a regime-conditional equity volatility estimate that can be used in option pricing
(see Notebook 06-Option-Pricing for the full CHMM pricing pipeline).

In [ ]:
# --- VIX REGIME VOLATILITY MAP ---
volatility_map = Dict{Int64, Float64}();
median_vix = Dict{Int64, Float64}();
count_per_state = Dict{Int64, Int64}();

for s in 1:K
    mask = findall(x -> x == s, decoded_states);
    count_per_state[s] = length(mask);
    if length(mask) > 0
        median_vix[s] = median(vix_close[mask]);
        volatility_map[s] = median_vix[s] / 100.0;
    else
        median_vix[s] = NaN;
        volatility_map[s] = 0.20;  # fallback
    end
end

println("\n" * "="^70);
println("  VIX Regime Volatility Map (K=$K)");
println("="^70);
println(rpad("State", 8) * rpad("# Days", 10) * rpad("Median VIX", 15) * rpad("Equity σ_s", 15) * rpad("μ (return)", 15));
println("-"^70);
for s in 1:K
    d = model.emission[s];
    println(
        rpad(s, 8) *
        rpad(count_per_state[s], 10) *
        rpad(round(median_vix[s], digits=2), 15) *
        rpad(round(volatility_map[s], digits=4), 15) *
        rpad(round(mean(d), digits=2), 15)
    );
end
println("="^70);
println("Interpretation: σ_s = median(VIX_close | state=s) / 100");

In [ ]:
# --- FIGURE: VOLATILITY MAP BAR CHART ---
colors_k = cgrad(:RdYlGn, K, categorical=true);

p_volmap = bar(1:K, [volatility_map[s] * 100 for s in 1:K],
    title="Regime-Conditional Equity Volatility from VIX (K=$K)", titlefontsize=10,
    xlabel="CHMM State", ylabel="Implied Equity Vol (%)",
    legend=false, color=colors_k[1:K], alpha=0.8,
    bar_width=0.6, size=(700, 400));

# Annotate with median VIX
for s in 1:K
    annotate!(p_volmap, s, volatility_map[s] * 100 + 1.0,
        text("VIX=$(round(median_vix[s], digits=1))", 7, :center));
end

display(p_volmap);
savefig(p_volmap, joinpath(_PATH_TO_FIGURES, "Fig-VIX-Volatility-Map-K$(K).svg"));

## Out-of-Sample Validation

We simulate paths of the same length as the OoS VIX data and compare distributional properties.
This tests whether the CHMM trained on historical data generalizes to the 2025 holdout period.

In [ ]:
# --- OOS SIMULATION ---
n_oos = length(R_vix_oos);

decoded_oos = Array{Float64,2}(undef, n_oos, N_PATHS);
for i in 1:N_PATHS
    s0 = rand(start_dist);
    states_oos = model(s0, n_oos);
    for j in 1:n_oos
        decoded_oos[j, i] = rand(model.emission[states_oos[j]]);
    end
end

println("OoS: $(N_PATHS) paths of length $(n_oos) simulated.");

In [ ]:
# --- OOS METRICS ---
metrics_oos = compute_vix_metrics(R_vix_oos, decoded_oos; L_acf=min(L, n_oos - 1));

println("\n" * "="^55);
println("  CHMM Out-of-Sample Validation: VIX (K=$K)");
println("="^55);
println("  KS pass rate (α=0.05):  $(metrics_oos.ks_rate)%");
println("  Excess kurtosis (sim):  $(metrics_oos.kurtosis)");
println("  Excess kurtosis (obs):  $(metrics_oos.kurt_obs)");
println("  ACF-MAE |Gₜ|:          $(metrics_oos.acf_mae)");
println("="^55);

In [ ]:
# --- FIGURE: OOS DENSITY + ACF ---
ci_oos = 2.576 / sqrt(n_oos);
τ_oos = 1:min(L, n_oos - 1);

# Panel (a): OoS density
p_oos_dens = plot(title="(a) OoS Marginal Density (KS: $(metrics_oos.ks_rate)%)",
    titlefontsize=9, xlabel="Excess Growth Rate", ylabel="Density");
histogram!(p_oos_dens, R_vix_oos, normalize=true, bins=50, alpha=0.3, color=:gray, label="OoS Observed");
density!(p_oos_dens, vec(decoded_oos), lw=2, color=:blue, alpha=0.7, label="CHMM (K=$K)");
xl_oos = quantile(R_vix_oos, 0.005);
xh_oos = quantile(R_vix_oos, 0.995);
xlims!(p_oos_dens, xl_oos, xh_oos);

# Panel (b): OoS ACF(|G|)
acf_oos_obs = autocor(abs.(R_vix_oos), τ_oos);
n_acf_oos = min(200, N_PATHS);
acf_oos_archive = hcat([autocor(abs.(decoded_oos[:, i]), τ_oos) for i in 1:n_acf_oos]...);
acf_oos_mean = mean(acf_oos_archive, dims=2)[:];

p_oos_acf = plot(τ_oos, acf_oos_obs, lw=2, color=:red, ls=:dash, label="OoS Observed",
    title="(b) OoS ACF(|Gₜ|)", titlefontsize=9,
    xlabel="Lag (trading days)", ylabel="ACF(|Gₜ|)");
plot!(p_oos_acf, τ_oos, acf_oos_mean, lw=2, color=:blue, label="CHMM mean");
hline!(p_oos_acf, [ci_oos, -ci_oos], lw=1, color=:gray, ls=:dash, label="99% CI");

# Panel (c): OoS Q-Q
probs_qq = range(0.01, 0.99, length=100);
q_obs_oos = quantile(R_vix_oos, probs_qq);
q_sim_oos = quantile(vec(decoded_oos), probs_qq);

p_oos_qq = plot(q_obs_oos, q_obs_oos, lw=2, color=:black, ls=:dash, label="Perfect",
    title="(c) OoS Q-Q Plot (1st-99th pctl)", titlefontsize=9,
    xlabel="Observed Quantiles", ylabel="Simulated Quantiles");
scatter!(p_oos_qq, q_obs_oos, q_sim_oos, ms=3, alpha=0.6, color=:blue, label="CHMM (K=$K)");

# Combine
fig_oos = plot(p_oos_dens, p_oos_acf, p_oos_qq, layout=(1, 3), size=(1400, 400),
    plot_title="VIX Out-of-Sample Validation (K=$K)",
    plot_titlefontsize=12);
display(fig_oos);

savefig(fig_oos, joinpath(_PATH_TO_FIGURES, "Fig-VIX-OoS-Comparison-K$(K).svg"));
savefig(fig_oos, joinpath(_PATH_TO_FIGURES, "Fig-VIX-OoS-Comparison-K$(K).pdf"));
println("Saved OoS comparison to figs/");

## VXX: Short-Term VIX Futures ETN

The **iPath Series B S&P 500 VIX Short-Term Futures ETN (VXX)** provides tradeable
exposure to volatility by tracking a rolling portfolio of first- and second-month
VIX futures contracts. Key characteristics:

- **Contango drag**: When the VIX futures term structure is in contango (upward-sloping),
  rolling from the expiring front-month to the next contract incurs a systematic cost.
  This produces a structural negative drift in VXX (~5--10% per month in calm markets).
  
- **Backwardation spikes**: During crises, the term structure inverts (backwardation),
  and VXX benefits from positive roll yield. These episodes are short but violent.

- **Mean reversion at the futures level**: Unlike VIX itself, VXX does not mean-revert
  to a level -- it trends toward zero over time due to contango drag (partially offset by
  periodic reverse splits).

### Applying CHMM to VXX

The same CHMM framework applies directly to VXX daily log returns:

```julia
# Example (once VXX data is added to the pipeline):
R_vxx = log_growth_matrix(vxx_dataset, "VXX"; Δt=1/252, keycol=:close);
model_vxx = build(MyContinuousHiddenMarkovModel, (
    observations = R_vxx,
    number_of_states = 6,
    max_iter = 60
));
```

The CHMM would capture the distinct regimes of VXX: 
- **Calm states**: Steady negative drift from contango roll cost, low volatility.
- **Transition states**: Moderate VIX movements, mixed roll yield.
- **Crisis states**: Large positive returns during vol spikes, backwardation.

**Note**: VXX price data is not currently included in the JLD2 data files. To add VXX
to the analysis pipeline, save a DataFrame with columns `(date, open, high, low, close, volume)`
into a JLD2 file and load it using the same `_jld2()` infrastructure. The return computation
and CHMM training workflow demonstrated above for VIX applies identically to VXX.

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice, investment recommendations, or trading strategies.